In [10]:
!pip install torch torchvision torchaudio

  Using cached torch-2.8.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached torchvision-0.23.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (6.1 kB)
  Using cached torchaudio-2.8.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (7.2 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cubl

In [11]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import cv2
from tqdm import tqdm
import random
import warnings
import json
warnings.filterwarnings('ignore')

In [17]:
class Config:
    """Configuration class for hyperparameters and settings"""
    def __init__(self):
        self.data_folder = "../maybe_useful/train_data"
        self.image_size = 512
        self.num_classes = 3  # background, lung, heart
        
        self.batch_size = 8
        self.num_epochs = 100
        self.early_stopping_patience = 10
        self.learning_rate = 0.001
        self.weight_decay = 1e-5
        self.bce_weight = 0.7
        self.jaccard_weight = 0.3

        self.enable_mask_cleaning = True
        
        self.train_ratio = 0.75
        self.val_ratio = 0.15
        self.test_ratio = 0.1
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.random_seed = 42
        self.model_save_path = "../maybe_useful/new_model.pth"

In [13]:
class ChestXrayDataset(Dataset):
    """Dataset class for chest X-ray images and masks"""
    
    def __init__(self, image_paths, mask_paths, transform=None, is_prediction=False, enable_preprocessing=False):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.is_prediction = is_prediction
        self.enable_preprocessing = enable_preprocessing  # New parameter for prediction preprocessing
    
    def __len__(self):
        return len(self.image_paths)
    
    def _preprocess_image(self, image):
        """Preprocess image: pad to square, then resize to 512x512, convert to grayscale"""
        # Convert to grayscale if not already
        if image.mode != 'L':
            image = image.convert('L')
        
        # Get current dimensions
        width, height = image.size
        
        # Pad to square
        max_dim = max(width, height)
        
        # Create new square image with black padding
        square_image = Image.new('L', (max_dim, max_dim), 0)  # 0 for black padding
        
        # Calculate position to paste original image (center it)
        paste_x = (max_dim - width) // 2
        paste_y = (max_dim - height) // 2
        
        # Paste original image onto square canvas
        square_image.paste(image, (paste_x, paste_y))
        
        # Resize to 512x512
        resized_image = square_image.resize((512, 512), Image.Resampling.LANCZOS)
        
        return resized_image
    
    def __getitem__(self, idx):
        # Load image
        image = Image.open(self.image_paths[idx])
        
        if self.enable_preprocessing:
            # Apply preprocessing for prediction (pad to square, then resize)
            image = self._preprocess_image(image)
        else:
            # Regular processing for training (direct resize - assumes already preprocessed)
            image = image.convert('L')  # Grayscale
            image = image.resize((512, 512))
        
        # Convert to tensor
        image = np.array(image, dtype=np.float32) / 255.0
        image = torch.tensor(image).unsqueeze(0)  # Add channel dimension
        
        if self.is_prediction:
            return image, self.image_paths[idx]
        
        # Load mask (only for training)
        mask = Image.open(self.mask_paths[idx]).convert('RGB')
        mask = mask.resize((512, 512))
        mask = np.array(mask)
        
        # Convert mask to class labels
        mask_labels = np.zeros((512, 512), dtype=np.int64)
        
        # Background (0,0,0) -> class 0
        mask_labels[(mask[:,:,0] == 0) & (mask[:,:,1] == 0) & (mask[:,:,2] == 0)] = 0
        
        # Left lung (85,85,85) and Right lung (170,170,170) -> class 1
        mask_labels[(mask[:,:,0] == 85) & (mask[:,:,1] == 85) & (mask[:,:,2] == 85)] = 1
        mask_labels[(mask[:,:,0] == 170) & (mask[:,:,1] == 170) & (mask[:,:,2] == 170)] = 1
        
        # Heart (255,255,255) -> class 2
        mask_labels[(mask[:,:,0] == 255) & (mask[:,:,1] == 255) & (mask[:,:,2] == 255)] = 2
        
        mask_labels = torch.tensor(mask_labels, dtype=torch.long)
        
        return image, mask_labels


class UNet(nn.Module):
    """Standard U-Net architecture for semantic segmentation"""
    
    def __init__(self, in_channels=1, out_channels=3):
        super(UNet, self).__init__()
        
        # Encoder (downsampling path)
        self.enc1 = self.double_conv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = self.double_conv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = self.double_conv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        
        self.enc4 = self.double_conv(256, 512)
        self.pool4 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = self.double_conv(512, 1024)
        
        # Decoder (upsampling path)
        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = self.double_conv(1024, 512)
        
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = self.double_conv(512, 256)
        
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = self.double_conv(256, 128)
        
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = self.double_conv(128, 64)
        
        # Final classifier
        self.classifier = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def double_conv(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool1(enc1))
        enc3 = self.enc3(self.pool2(enc2))
        enc4 = self.enc4(self.pool3(enc3))
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool4(enc4))
        
        # Decoder
        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.dec4(dec4)
        
        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.dec3(dec3)
        
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.dec2(dec2)
        
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.dec1(dec1)
        
        return self.classifier(dec1)


class JaccardLoss(nn.Module):
    """Jaccard (IoU) loss for segmentation"""
    
    def __init__(self, smooth=1e-6):
        super(JaccardLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, predictions, targets, num_classes=3):
        # Apply softmax to predictions
        predictions = torch.softmax(predictions, dim=1)
        
        # Convert targets to one-hot encoding
        targets_one_hot = torch.zeros_like(predictions)
        targets_one_hot.scatter_(1, targets.unsqueeze(1), 1)
        
        jaccard_loss = 0
        for i in range(num_classes):
            pred_i = predictions[:, i]
            target_i = targets_one_hot[:, i]
            
            intersection = torch.sum(pred_i * target_i, dim=(1, 2))
            union = torch.sum(pred_i, dim=(1, 2)) + torch.sum(target_i, dim=(1, 2)) - intersection
            
            jaccard = (intersection + self.smooth) / (union + self.smooth)
            jaccard_loss += (1 - jaccard).mean()
        
        return jaccard_loss / num_classes


class CombinedLoss(nn.Module):
    """Combined BCE and Jaccard loss"""
    
    def __init__(self, bce_weight=0.7, jaccard_weight=0.3):
        super(CombinedLoss, self).__init__()
        self.bce_weight = bce_weight
        self.jaccard_weight = jaccard_weight
        self.bce_loss = nn.CrossEntropyLoss()
        self.jaccard_loss = JaccardLoss()
    
    def forward(self, predictions, targets):
        bce = self.bce_loss(predictions, targets)
        jaccard = self.jaccard_loss(predictions, targets)
        return self.bce_weight * bce + self.jaccard_weight * jaccard

In [14]:
def modify_json_from_csv(input_json_path, csv_path, output_json_path):
    # Read the input JSON file
    with open(input_json_path, 'r') as f:
        data = json.load(f)
    
    # Read the CSV file
    df = pd.read_csv(csv_path)
    
    # Create predictions list
    predictions = []
    for _, row in df.iterrows():
        filename = row['name']
        ctr = row['CTR']
        # Determine class based on CTR
        class_value = 1 if ctr > 0.5 else 0
        # Format prediction entry
        prediction = {
            "filename": filename,
            "a": 0,
            "b": 0,
            "c": 0,
            "ratio": round(ctr, 5),
            "class": class_value
        }
        predictions.append(prediction)
    
    # Update the JSON data with predictions
    data['tahminler'] = predictions
    
    # Save the modified JSON to a new file
    with open(output_json_path, 'w') as f:
        json.dump(data, f, indent=4)

def calculate_ctr_from_mask(mask):
    """Calculate CTR from hard segmentation mask (numpy array)"""
    # Heart (Class 2)
    heart_mask = (mask == 2)
    heart_coords = np.where(heart_mask)
    cardiac_width = np.max(heart_coords[1]) - np.min(heart_coords[1]) + 1 if len(heart_coords[1]) > 0 else 0
    
    # Lung (Class 1)
    lung_mask = (mask == 1)
    lung_coords = np.where(lung_mask)
    thoracic_width = 0
    if len(lung_coords[0]) > 0:
        all_y = np.unique(lung_coords[0])
        max_distance = 0
        for y in all_y:
            x_vals = lung_coords[1][lung_coords[0] == y]
            if len(x_vals) > 0:
                width = np.max(x_vals) - np.min(x_vals) + 1
                max_distance = max(max_distance, width)
        thoracic_width = max_distance
    
    return cardiac_width / thoracic_width if thoracic_width > 0 else 0.0


def calculate_iou(pred_mask, true_mask):
    """Calculate IoU for segmentation evaluation"""
    # Calculate pixel-wise accuracy as IoU (since we treat the whole mask as one entity)
    correct_pixels = (pred_mask == true_mask)
    total_pixels = pred_mask.size
    iou = np.sum(correct_pixels) / total_pixels
    return iou


def clean_mask(mask_np):
    """
    Cleans a segmentation mask by filling holes and keeping the largest components.
    - For lungs (class 1), keeps the two largest components.
    - For heart (class 2), keeps the single largest component.
    """
    cleaned_mask = np.zeros_like(mask_np, dtype=np.uint8)
    
    # Process each class (lungs and heart)
    for class_id, num_components_to_keep in [(1, 2), (2, 1)]: # (class_id, num_components)
        # Create a binary mask for the current class
        binary_mask = (mask_np == class_id).astype(np.uint8)
        
        # Skip if the class is not present in the mask
        if np.sum(binary_mask) == 0:
            continue
            
        # --- Step 1: Fill holes ---
        # Find external contours and draw them filled to close internal holes
        contours, hierarchy = cv2.findContours(binary_mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
        for i, contour in enumerate(contours):
            # Check if it's a top-level contour (i.e., not a hole)
            if hierarchy[0][i][3] == -1:
                cv2.drawContours(binary_mask, [contour], 0, 255, -1)

        # --- Step 2: Keep the largest components ---
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
        
        if num_labels > 1: # Check if there are any components besides the background
            # Get the area of each component, excluding the background (label 0)
            areas = stats[1:, cv2.CC_STAT_AREA]
            
            # Sort component indices by area in descending order
            component_indices = np.argsort(areas)[::-1]
            
            # Create a new mask to store the final components
            final_class_mask = np.zeros_like(binary_mask)
            
            # Keep the top N components
            for i in range(min(num_components_to_keep, len(component_indices))):
                component_idx = component_indices[i] + 1 # Add 1 to match label index
                final_class_mask[labels == component_idx] = 1
            
            # Add the cleaned class mask to the final combined mask
            cleaned_mask[final_class_mask == 1] = class_id
            
    return cleaned_mask

def set_random_seeds(seed):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

In [15]:
class UNetSegmentationModel:
    """Main model class for U-Net segmentation"""
    
    def __init__(self, config=None):
        self.config = config if config is not None else Config()
        set_random_seeds(self.config.random_seed)
        
        self.model = UNet(in_channels=1, out_channels=self.config.num_classes)
        self.model.to(self.config.device)
        
        self.criterion = CombinedLoss(
            bce_weight=self.config.bce_weight,
            jaccard_weight=self.config.jaccard_weight
        )
        self.optimizer = optim.Adam(
            self.model.parameters(),
            lr=self.config.learning_rate,
            weight_decay=self.config.weight_decay
        )
    
    def prepare_data(self):
        """Prepare and split the dataset"""
        images_folder = os.path.join(self.config.data_folder, "images")
        masks_folder = os.path.join(self.config.data_folder, "masks")
        
        image_files = [f for f in os.listdir(images_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        image_paths = []
        mask_paths = []
        
        for img_file in image_files:
            img_path = os.path.join(images_folder, img_file)
            mask_path = os.path.join(masks_folder, img_file)
            
            if os.path.exists(mask_path):
                image_paths.append(img_path)
                mask_paths.append(mask_path)
        
        # Split dataset
        train_img, temp_img, train_mask, temp_mask = train_test_split(
            image_paths, mask_paths, 
            test_size=(1 - self.config.train_ratio),
            random_state=self.config.random_seed
        )
        
        val_img, test_img, val_mask, test_mask = train_test_split(
            temp_img, temp_mask,
            test_size=self.config.test_ratio / (self.config.val_ratio + self.config.test_ratio),
            random_state=self.config.random_seed
        )
        
        self.train_dataset = ChestXrayDataset(train_img, train_mask)
        self.val_dataset = ChestXrayDataset(val_img, val_mask)
        self.test_dataset = ChestXrayDataset(test_img, test_mask)
        
        self.train_loader = DataLoader(self.train_dataset, batch_size=self.config.batch_size, shuffle=True)
        self.val_loader = DataLoader(self.val_dataset, batch_size=self.config.batch_size, shuffle=False)
        self.test_loader = DataLoader(self.test_dataset, batch_size=1, shuffle=False)
        
        print(f"Dataset split - Train: {len(self.train_dataset)}, Val: {len(self.val_dataset)}, Test: {len(self.test_dataset)}")
    
    def _calculate_epoch_iou(self, loader):
        """Calculate the mean IoU for an entire epoch."""
        self.model.eval()
        total_correct = 0
        total_pixels = 0
        with torch.no_grad():
            for images, masks in loader:
                images = images.to(self.config.device)
                masks = masks.to(self.config.device)
                
                outputs = self.model(images)
                preds = torch.argmax(outputs, dim=1)
                
                total_correct += (preds == masks).sum().item()
                total_pixels += masks.numel()
        
        mean_iou = total_correct / total_pixels
        return mean_iou

    def train(self):
        """Train the model with early stopping"""
        print("Starting training...")
        best_val_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(self.config.num_epochs):
            # Training phase
            self.model.train()
            train_loss = 0.0
            
            for images, masks in tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{self.config.num_epochs} [Train]"):
                images = images.to(self.config.device)
                masks = masks.to(self.config.device)
                
                self.optimizer.zero_grad()
                outputs = self.model(images)
                loss = self.criterion(outputs, masks)
                loss.backward()
                self.optimizer.step()
                
                train_loss += loss.item()
            
            # Validation phase
            self.model.eval()
            val_loss = 0.0
            
            with torch.no_grad():
                for images, masks in tqdm(self.val_loader, desc=f"Epoch {epoch+1}/{self.config.num_epochs} [Val]"):
                    images = images.to(self.config.device)
                    masks = masks.to(self.config.device)
                    
                    outputs = self.model(images)
                    loss = self.criterion(outputs, masks)
                    val_loss += loss.item()
            
            train_loss /= len(self.train_loader)
            val_loss /= len(self.val_loader)
            
            # Calculate IoU for both training and validation sets
            train_iou = self._calculate_epoch_iou(self.train_loader)
            val_iou = self._calculate_epoch_iou(self.val_loader)
            
            print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train IoU: {train_iou:.4f} | "
                  f"Val Loss: {val_loss:.4f}, Val IoU: {val_iou:.4f}")
            
            # Early stopping and model saving logic
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                self.save_model()
                print(f"Validation loss improved. Best model saved with val loss: {val_loss:.4f}")
            else:
                patience_counter += 1
                print(f"Validation loss did not improve. Patience: {patience_counter}/{self.config.early_stopping_patience}")

            if patience_counter >= self.config.early_stopping_patience:
                print("Early stopping triggered.")
                break
        
        print("Training completed!")
        
    def evaluate(self):
        """Evaluate the model on test set"""
        print("Evaluating model...")
        self.model.eval()
        
        all_iou = []
        all_ctr_pred = []
        all_ctr_true = []
        all_cardiomegaly_pred = []
        all_cardiomegaly_true = []
        
        with torch.no_grad():
            for images, masks in tqdm(self.test_loader, desc="Evaluating"):
                images = images.to(self.config.device)
                masks = masks.to(self.config.device)
                
                outputs = self.model(images)
                pred_masks = torch.argmax(outputs, dim=1)
                
                # Convert to numpy for evaluation
                pred_mask_np = pred_masks.cpu().numpy()[0]
                true_mask_np = masks.cpu().numpy()[0]
                
                # Conditionally clean the predicted mask
                if self.config.enable_mask_cleaning:
                    pred_mask_np = clean_mask(pred_mask_np)
                
                # Calculate IoU
                iou = calculate_iou(pred_mask_np, true_mask_np)
                all_iou.append(iou)
                
                # Calculate CTR
                ctr_pred = calculate_ctr_from_mask(pred_mask_np)
                ctr_true = calculate_ctr_from_mask(true_mask_np)
                all_ctr_pred.append(ctr_pred)
                all_ctr_true.append(ctr_true)
                
                # Calculate cardiomegaly classification
                cardiomegaly_pred = 1 if ctr_pred > 0.5 else 0
                cardiomegaly_true = 1 if ctr_true > 0.5 else 0
                all_cardiomegaly_pred.append(cardiomegaly_pred)
                all_cardiomegaly_true.append(cardiomegaly_true)
        
        # Calculate metrics
        mean_iou = np.mean(all_iou)
        
        # 1-MAPE for CTR regression
        mape = np.mean(np.abs((np.array(all_ctr_true) - np.array(all_ctr_pred)) / (np.array(all_ctr_true) + 1e-8))) * 100
        one_minus_mape = 1 - (mape / 100)
        
        # F1 score for cardiomegaly classification
        f1 = f1_score(all_cardiomegaly_true, all_cardiomegaly_pred)
        
        print(f"\nEvaluation Results:")
        print(f"IoU: {mean_iou:.4f}")
        print(f"1-MAPE (CTR): {one_minus_mape:.4f}")
        print(f"F1 Score (Cardiomegaly): {f1:.4f}")
        
        return mean_iou, one_minus_mape, f1
    
    def predict(self, images_folder, output_csv_path="predictions.csv"):
        """Predict on unlabeled images with preprocessing and save results to CSV"""
        print(f"Predicting on images in {images_folder} with preprocessing...")
        self.model.eval()
        
        image_files = [f for f in os.listdir(images_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        image_paths = [os.path.join(images_folder, f) for f in image_files]
        
        # Enable preprocessing for prediction
        prediction_dataset = ChestXrayDataset(
            image_paths, 
            None, 
            is_prediction=True, 
            enable_preprocessing=True  # Enable preprocessing for predictions
        )
        prediction_loader = DataLoader(prediction_dataset, batch_size=1, shuffle=False)
        
        results = []
        
        with torch.no_grad():
            for images, img_paths in tqdm(prediction_loader, desc="Predicting"):
                images = images.to(self.config.device)
                
                outputs = self.model(images)
                pred_masks = torch.argmax(outputs, dim=1)
                pred_mask_np = pred_masks.cpu().numpy()[0]
    
                if self.config.enable_mask_cleaning:
                    pred_mask_np = clean_mask(pred_mask_np)
                
                ctr = calculate_ctr_from_mask(pred_mask_np)
                cardiomegaly = 1 if ctr > 0.5 else 0
                
                # Keep the full filename with extension
                filename = os.path.basename(img_paths[0])
                results.append({
                    'name': filename,
                    'CTR': ctr,
                    'cardiomegaly': cardiomegaly
                })
        
        df = pd.DataFrame(results)
        df.to_csv(output_csv_path, index=False)
        print(f"Predictions saved to {output_csv_path}")
        
        return df
    
    def evaluate_external_dataset(self, images_folder, data_csv_path):
        """Evaluate on external dataset with only CTR labels, matching filenames including extensions"""
        print(f"Evaluating external dataset...")
        
        df_true = pd.read_csv(data_csv_path)
        
        # Use the updated predict method with preprocessing
        df_pred = self.predict(images_folder, "temp_predictions.csv")
        
        # Merge by full filename with extension
        df_merged = df_pred.merge(df_true, on='name', suffixes=('_pred', '_true'))
        
        ctr_pred = df_merged['CTR_pred'].values
        ctr_true = df_merged['CTR_true'].values
        
        valid_indices = ctr_true > 1e-6
        if np.sum(valid_indices) > 0:
            ctr_pred_valid = ctr_pred[valid_indices]
            ctr_true_valid = ctr_true[valid_indices]
            mape = np.mean(np.abs((ctr_true_valid - ctr_pred_valid) / ctr_true_valid)) * 100
            one_minus_mape = 1 - (mape / 100)
        else:
            print("Warning: No valid CTR values for MAPE calculation")
            one_minus_mape = 0.0
        
        cardiomegaly_pred = (ctr_pred > 0.5).astype(int)
        cardiomegaly_true = (ctr_true > 0.5).astype(int)
        
        if len(np.unique(cardiomegaly_pred)) == 1 or len(np.unique(cardiomegaly_true)) == 1:
            print("Warning: All predictions or ground truth are the same class")
            from sklearn.metrics import accuracy_score
            accuracy = accuracy_score(cardiomegaly_true, cardiomegaly_pred)
            print(f"Using accuracy instead of F1: {accuracy:.4f}")
            f1 = accuracy
        else:
            f1 = f1_score(cardiomegaly_true, cardiomegaly_pred)
        
        print(f"\nExternal Dataset Evaluation Results:")
        print(f"1-MAPE (CTR): {one_minus_mape:.4f}")
        print(f"F1 Score (Cardiomegaly): {f1:.4f}")
        
        if os.path.exists("temp_predictions.csv"):
            os.remove("temp_predictions.csv")
        
        return one_minus_mape, f1

    def save_model(self):
        """Save the trained model"""
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'config': self.config
        }, self.config.model_save_path)

    def load_model(self, model_path=None):
        """Load a trained model"""
        if model_path is None:
            model_path = self.config.model_save_path
        
        checkpoint = torch.load(model_path, map_location=self.config.device, weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        print(f"Model loaded from {model_path}")

In [18]:
config = Config()
model = UNetSegmentationModel(config)

In [19]:
model.prepare_data()

Dataset split - Train: 815, Val: 163, Test: 109


In [ ]:
model.train()

Starting training...


Epoch 1/100 [Train]:   0%|          | 0/102 [00:00<?, ?it/s]

In [8]:
model.load_model("model.pth")

Model loaded from model.pth


In [141]:
model.evaluate()

Evaluating model...


Evaluating: 100%|██████████| 39/39 [00:05<00:00,  7.11it/s]


Evaluation Results:
IoU: 0.9786
1-MAPE (CTR): 0.9767
F1 Score (Cardiomegaly): 0.9091


(0.9786321199857272, 0.9766886440453048, 0.9090909090909091)

In [ ]:
model.evaluate_external_dataset("../maybe_useful/external_test_data/images", "../maybe_useful/external_test_data/data.csv")

Evaluating external dataset...
Predicting on images in /kaggle/input/test-tekno/images with preprocessing...


Predicting: 100%|██████████| 102/102 [00:07<00:00, 13.46it/s]

Predictions saved to temp_predictions.csv

External Dataset Evaluation Results:
1-MAPE (CTR): 0.9447
F1 Score (Cardiomegaly): 0.9016


(0.9447058792870323, 0.9016393442622951)

In [9]:
model.predict("test_images", "predictions.csv")

Predicting on images in test_images with preprocessing...


Predicting: 100%|██████████| 102/102 [03:44<00:00,  2.20s/it]

Predictions saved to predictions.csv


,name,CTR,cardiomegaly
0,1 (19).jpg,0.571429,1
1,1 (62).jpg,0.378788,0
2,1 (24).jpg,0.472500,0
3,1 (33).jpg,0.592834,1
4,1 (68).jpg,0.472622,0
...,...,...,...
97,1 (38).jpg,0.579251,1
98,1 (77).jpg,0.524887,1
99,1 (18).jpg,0.617080,1
100,1 (66).jpg,0.479218,0


In [10]:
modify_json_from_csv("cxr_input.json", "predictions.csv", "output.json")